# Day 079 — Exercise 1: Tools and the Registry

**What you'll build:** a safe calculator, a tool registry, and a function that renders the registry into prompt text.

**Why it matters:** an agent is an LLM plus a loop plus **tools**. The tool registry is how the agent knows what it can do. A calculator built on `eval()` would let model output run arbitrary code — so we parse arithmetic with `ast` instead.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing
    a runaway loop (a model that never says 'finish').
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn


## Task

1. `safe_calculate(expression) -> number` — `ast.parse(expression, mode='eval')`, then recursively evaluate only `BinOp`/`UnaryOp`/numeric `Constant` nodes using an `_OPS` dict `{ast.Add: operator.add, ...}`. Raise `ValueError` on anything else.
2. `DEFAULT_TOOLS` — a dict `{name: {'description', 'parameters', 'fn'}}` with a `calculator` (`fn = lambda args: str(safe_calculate(args['expression']))`) and a `word_count` (`fn = lambda args: str(len(str(args['text']).split()))`).
3. `build_tool_descriptions(tools) -> str` — one line per tool: `- name(params): description`, joined with newlines.

## Your Implementation

In [ ]:
import ast, operator

def safe_calculate(expression):
    """Evaluate arithmetic (+ - * / ** %, parens) without eval()."""
    raise NotImplementedError

DEFAULT_TOOLS = {}  # calculator + word_count

def build_tool_descriptions(tools):
    """Render the registry as a text block for the prompt."""
    raise NotImplementedError


In [ ]:
import ast
import json
import operator

# ── a safe calculator tool (no eval) ─────────────────────────────────────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    """Recursively evaluate an arithmetic AST node. Raises on anything unsafe."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate a basic arithmetic expression without eval().

    Supports + - * / ** % and parentheses. Anything else (names, calls,
    attribute access) raises ValueError. This is the safe way to give an
    agent a calculator: never eval() untrusted model output.
    """
    tree = ast.parse(expression, mode="eval")
    return _eval_node(tree.body)


# ── the tool registry ────────────────────────────────────────────────────────
# A tool = {description, parameters, fn}. fn takes an args dict, returns a str.
DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "word_count": {
        "description": "Count the words in a piece of text.",
        "parameters": {"text": "string - the text to count words in"},
        "fn": lambda args: str(len(str(args["text"]).split())),
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as a text block for the prompt."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)


## Automated checks

In [ ]:

score, total = 0, 5
try:
    assert safe_calculate('2 + 3') == 5
    assert safe_calculate('2 * (3 + 4)') == 14
    assert safe_calculate('10 / 4') == 2.5
    score += 1; print("✅ safe_calculate handles arithmetic")

    raised = False
    try:
        safe_calculate('__import__("os").system("echo hi")')
    except Exception:
        raised = True
    assert raised, "safe_calculate should reject non-arithmetic input"
    score += 1; print("✅ safe_calculate rejects non-arithmetic (no eval)")

    assert 'calculator' in DEFAULT_TOOLS and 'word_count' in DEFAULT_TOOLS
    score += 1; print("✅ DEFAULT_TOOLS has calculator and word_count")

    assert DEFAULT_TOOLS['calculator']['fn']({'expression': '6 * 7'}) == '42'
    assert DEFAULT_TOOLS['word_count']['fn']({'text': 'a b c'}) == '3'
    score += 1; print("✅ tool fns run and return strings")

    desc = build_tool_descriptions(DEFAULT_TOOLS)
    assert 'calculator' in desc and 'word_count' in desc
    score += 1; print("✅ build_tool_descriptions lists every tool")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
import ast
import json
import operator

# ── a safe calculator tool (no eval) ─────────────────────────────────────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    """Recursively evaluate an arithmetic AST node. Raises on anything unsafe."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate a basic arithmetic expression without eval().

    Supports + - * / ** % and parentheses. Anything else (names, calls,
    attribute access) raises ValueError. This is the safe way to give an
    agent a calculator: never eval() untrusted model output.
    """
    tree = ast.parse(expression, mode="eval")
    return _eval_node(tree.body)


# ── the tool registry ────────────────────────────────────────────────────────
# A tool = {description, parameters, fn}. fn takes an args dict, returns a str.
DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "word_count": {
        "description": "Count the words in a piece of text.",
        "parameters": {"text": "string - the text to count words in"},
        "fn": lambda args: str(len(str(args["text"]).split())),
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as a text block for the prompt."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)
```

**Why `ast` instead of `eval`?** `eval('__import__("os").system(...)')` runs arbitrary code. Walking the AST and only allowing arithmetic nodes means the worst a malicious expression can do is raise `ValueError`.

</details>